# 환경설정

In [2]:
# 라이브러리 설치
# pip install langchain-openai langchain-core langgraph langchain-chroma rank_bm25

import re
import os
import logging
import json
import pymongo
from pymongo import MongoClient
from datetime import datetime
from dotenv import load_dotenv
import pandas as pd
from typing import Literal, TypedDict, List
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain.schema import Document
from langchain_community.retrievers import BM25Retriever
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain.retrievers import EnsembleRetriever
from langgraph.graph import StateGraph, START, END
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.retrievers import ContextualCompressionRetriever
from langchain.text_splitter import RecursiveCharacterTextSplitter

python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 6


In [3]:
# 환경 변수 로드 및 로깅 설정
load_dotenv()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# 로그 저장 디렉토리 설정
LOG_DIR = "chat_logs"
os.makedirs(LOG_DIR, exist_ok=True)

"""# MongoDB 클라이언트 설정
MONGO_IP = os.getenv("MONGO_IP")
MONGO_PORT = int(os.getenv("MONGO_PORT"))
MONGO_USER = os.getenv("MONGO_USER")
MONGO_PASSWORD = os.getenv("MONGO_PASSWORD")

# 연결 URI 생성
mongo_uri = f"mongodb://{MONGO_USER}:{MONGO_PASSWORD}@{MONGO_IP}:{MONGO_PORT}/?authSource=admin"
client = MongoClient(mongo_uri)

# 사용할 데이터베이스와 컬렉션 지정
db = client['chatbot_db']
collection = db['chat_logs']
"""

python-dotenv could not parse statement starting at line 6


'# MongoDB 클라이언트 설정\nMONGO_IP = os.getenv("MONGO_IP")\nMONGO_PORT = int(os.getenv("MONGO_PORT"))\nMONGO_USER = os.getenv("MONGO_USER")\nMONGO_PASSWORD = os.getenv("MONGO_PASSWORD")\n\n# 연결 URI 생성\nmongo_uri = f"mongodb://{MONGO_USER}:{MONGO_PASSWORD}@{MONGO_IP}:{MONGO_PORT}/?authSource=admin"\nclient = MongoClient(mongo_uri)\n\n# 사용할 데이터베이스와 컬렉션 지정\ndb = client[\'chatbot_db\']\ncollection = db[\'chat_logs\']\n'

In [4]:
#----1. 모델 정의----
model = ChatOpenAI(
    model_name='gpt-4o-mini',
    temperature=0
)

# 1. 전처리1

### 1-1. txt -> xlsx

In [5]:
# txt -> df

import pandas as pd
import os

# 원본 데이터가 있는 상위 폴더 경로
root_folder_path = "/Users/sdyplum/Desktop/DSCAP/Whatsapp_Crawling_Data/All_chats/Chats"

# 엑셀 파일을 저장할 폴더 경로
output_folder_path = "/Users/sdyplum/Desktop/DSCAP/DataScience_Capstone/data_preprocessing"

# 1. os.walk()로 상위 폴더부터 모든 하위 폴더 순회
# root: 현재 순회 중인 폴더 경로
# dirs: 현재 폴더에 있는 하위 폴더들 리스트
# files: 현재 폴더에 있는 파일들 리스트

for root, dirs, files in os.walk(root_folder_path):
    data = [] # 현재 폴더의 데이터를 담을 리스트 (매번 초기화)
    
    # 3. 현재 폴더의 파일들을 순회합니다.
    for file_name in files:
        if file_name.endswith(".txt"):
            file_path = os.path.join(root, file_name)
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    content = f.read()
                data.append([file_name, content])
            except Exception as e:
                print(f"파일 읽기 오류 '{file_path}': {e}")
    
    # 4. 현재 폴더에 처리할 txt 파일이 있었다면 데이터프레임 생성 및 저장
    if data:
        # 현재 폴더 이름을 가져오기
        current_folder_name = os.path.basename(root)
        
        # 데이터프레임 생성
        df = pd.DataFrame(data, columns=["file_name", "content"])
        
        # 저장할 엑셀 파일 경로 설정
        output_file_path = os.path.join(output_folder_path, f"{current_folder_name}.xlsx")
        
        # 엑셀 파일로 저장
        df.to_excel(output_file_path, index=False, engine="openpyxl")
        print(f"'{current_folder_name}' 폴더의 txt 파일들을 '{current_folder_name}.xlsx'로 저장 완료")

'whatsapp_exports_Navid' 폴더의 txt 파일들을 'whatsapp_exports_Navid.xlsx'로 저장 완료
'whatsapp_exports_Hadi' 폴더의 txt 파일들을 'whatsapp_exports_Hadi.xlsx'로 저장 완료
'whatsapp_exports_Hassan' 폴더의 txt 파일들을 'whatsapp_exports_Hassan.xlsx'로 저장 완료
'whatsapp_exports_Jalral' 폴더의 txt 파일들을 'whatsapp_exports_Jalral.xlsx'로 저장 완료
'whatsapp_exports_Haroon' 폴더의 txt 파일들을 'whatsapp_exports_Haroon.xlsx'로 저장 완료
'whatsapp_exports_Anjila' 폴더의 txt 파일들을 'whatsapp_exports_Anjila.xlsx'로 저장 완료
'whatsapp_exports_Omid' 폴더의 txt 파일들을 'whatsapp_exports_Omid.xlsx'로 저장 완료
'whatsapp_exports_Bhram' 폴더의 txt 파일들을 'whatsapp_exports_Bhram.xlsx'로 저장 완료


In [6]:
cwd = os.getcwd()

folder_path = cwd
output_file = os.path.join(cwd, 'merged_files.xlsx')

# 1. .xlsx 확장자를 가진 모든 파일 목록 가져오기
all_files = os.listdir(folder_path)
excel_files = [f for f in all_files if f.endswith('.xlsx')]

if not excel_files:
    print("지정된 폴더에 엑셀 파일(.xlsx)이 없습니다.")
else:
    print(f"총 {len(excel_files)}개의 엑셀 파일을 병합합니다.")
    
    # 2. 각 엑셀 파일을 순서대로 읽어 데이터프레임 리스트에 추가
    df_list = []
    for file_name in excel_files:
        file_path = os.path.join(folder_path, file_name)
        df = pd.read_excel(file_path)
        df_list.append(df)
        print(f" - '{file_name}' 파일 로드 완료")

    # 3. 데이터프레임 리스트를 하나로 합치기
    merged_df = pd.concat(df_list, ignore_index=True) # ignore_index=True 각 파일의 기존 인덱스를 무시하고 새로 인덱스를 부여

    # 4. 병합된 데이터프레임을 새로운 엑셀 파일로 저장합니다.
    merged_df.to_excel(output_file, index=False, engine='openpyxl') # index=False 데이터프레임의 인덱스를 엑셀 파일에 쓰지 않도록 합니다.

    print(f"결과가 '{output_file}' 파일에 저장되었습니다.")

총 8개의 엑셀 파일을 병합합니다.
 - 'whatsapp_exports_Anjila.xlsx' 파일 로드 완료
 - 'whatsapp_exports_Navid.xlsx' 파일 로드 완료
 - 'whatsapp_exports_Haroon.xlsx' 파일 로드 완료
 - 'whatsapp_exports_Bhram.xlsx' 파일 로드 완료
 - 'whatsapp_exports_Jalral.xlsx' 파일 로드 완료
 - 'whatsapp_exports_Hadi.xlsx' 파일 로드 완료
 - 'whatsapp_exports_Omid.xlsx' 파일 로드 완료
 - 'whatsapp_exports_Hassan.xlsx' 파일 로드 완료
결과가 '/Users/sdyplum/Desktop/DSCAP/DataScience_Capstone/data_preprocessing/merged_files.xlsx' 파일에 저장되었습니다.


### 1-2. 시스템 메시지 + 이모티콘 제거

In [ ]:
# (필요시) 데이터 불러오기
df = pd.read_excel("merged_files.xlsx")

def clean_chat_text(text: str) -> str:
    # 1. 불필요한 안내 문구 제거
    text = re.sub(r"Messages and calls are end-to-end encrypted.*?\n", "", text)
    text = re.sub(r"Welcome to the chat:.*?\n", "", text)

    # 2. 특수문자/이모지 제거
    text = re.sub(r"[^\w\s\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF]", " ", text)  
    # (아랍/파슈토/다리어 문자 범위는 보존)

    return text

# content 열 전처리 적용
df["clean_content"] = df["content"].apply(clean_chat_text)

# 중복 제거
df = df.drop_duplicates(subset=["clean_content"]).reset_index(drop=True)

# 저장
df.to_excel("merged_cleaned.xlsx", index=False, engine="openpyxl")

### 1-3. 기계적인 메시지 삭제

In [ ]:
# 삭제할 키워드 목록
unwanted_keywords = [
    "wds-ill-ads-WA.st0",
    "chat-filled-refreshed2",
    "megaphone-refreshed-32"
]

# 삭제할 키워드들을 '|'로 묶어 하나의 검색 패턴(정규식)으로 만듦
search_pattern = '|'.join(unwanted_keywords)

# 모든 행을 검사하여 unwanted_keywords가 포함된 행 서치
df_filtered = df[~df.apply(lambda x: x.astype(str).str.contains(search_pattern)).any(axis=1)]

print("--- 원본 데이터프레임 (df) ---")
print(df)
print("\n--- 필터링 후 데이터프레임 (df_filtered) ---")
print(df_filtered)

--- 원본 데이터프레임 (df1) ---
                  file_name  \
0       _93 77 594 1073.txt   
1       _93 74 967 6191.txt   
2       _93 79 795 2761.txt   
3       _93 78 811 6598.txt   
4       _93 78 653 4544.txt   
...                     ...   
3479    _93 79 296 9055.txt   
3480    _93 78 926 2123.txt   
3481    _93 74 982 7240.txt   
3482  _90 536 307 11 09.txt   
3483    _93 70 663 4501.txt   

                                                content  \
0     ✨+93 77 594 1073✨\nMessages and calls are end-...   
1     ✨+93 74 967 6191✨\nMessages and calls are end-...   
2     ✨+93 79 795 2761✨\nMessages and calls are end-...   
3     ✨+93 78 811 6598✨\nMessages and calls are end-...   
4     ✨+93 78 653 4544✨\nMessages and calls are end-...   
...                                                 ...   
3479  ✨+93 79 296 9055✨\nMessages and calls are end-...   
3480  ✨+93 78 926 2123✨\nMessages and calls are end-...   
3481  ✨+93 74 982 7240✨\nMessages and calls are end-...   
3482  ✨+90 53

### 1-4. AI 활용하여 사적인 대화 판별 후 삭제

In [10]:
df1 = pd.read_excel("merged_cleaned.xlsx")
data = df1

system = """
당신은 데이터 감별사입니다. 입력받은 파일의 데이터 중 사적인 데이터를 감별하여 반환합니다. 
사적인 대화는 모르는 사람과 대화하는 것이 아닌, 친근한 대화를 의미합니다.
사적인 대화라고 판단된다면 해당하는 행의 번호를 반환합니다. 
"""

prompt = ChatPromptTemplate.from_messages([("system", system), ("human", "{data}")])
chain = prompt | model | StrOutputParser()

out = chain.invoke({"data": data})
out

2025-09-12 22:26:18,036 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


'사적인 대화라고 판단되는 행의 번호는 다음과 같습니다:\n\n- 3480\n- 3481\n- 3483'